# FASTT — main results on VoxPopuli

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/main_voxpopuli.ipynb)

This notebook reproduces the paper's main experiment on VoxPopuli. Every method's hyperparameters are selected on
a 25 % hold-out of the corpus; every method is then trained and tested on the same ten folds of a 5×2
cross-validation of the remaining clips, and the two FASTT variants are compared with every other method on those
folds.

The configuration is fixed in `fastt.configs.MAIN["voxpopuli"]` and the code is in `fastt.experiment`. The
features and the stored searches and folds are read from the Hugging Face dataset
[`huseyin-karaca/fastt`](https://huggingface.co/datasets/huseyin-karaca/fastt); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold refitted from the published features (`RETUNE = True`: the search as well) | a CPU, hours |
| **3** | level 2 on features rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`FASTT_HUB` names another place) | a GPU for `extract` |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.configs import MAIN
from fastt.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again

exp = MainExperiment(MAIN["voxpopuli"], level=LEVEL, retune=RETUNE)

VoxPopuli: level 1 — search read from the Hub, folds read from the Hub
experts  ESPnet Conformer-RNN-T, ESPnet Transformer, data2vec-audio-base
arms     FASTT-Alternating, FASTT-Boosted, Raw XGBoost (AdaSTT), Raw SDT, SelectKBest XGBoost, SelectKBest SDT, ROVER (weighted), ROVER (confidence-weighted), CN-MBR, Score-Fusion, Hard-Selector (plain CE), Hard-Selector (cost-sensitive CE), Soft-Selector (mixture), QE-Router


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every method and their choices; every other parameter is fixed
(`fastt.configs.SPACES`). The spaces are the same on every corpus.

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed
  [FASTT-Alternating  (18 free of 18)]
           parameter kind                                   choices / range
      transform_type  cat {nonlinear, linear, diagonal, low_rank, identity}
      num_iterations  cat                                     {6, 12, 3, 1}
        transform_lr  cat                             {0.001, 0.0001, 0.01}
     transform_steps  cat                                    {100, 25, 400}
        surrogate_lr  cat                                {0.01, 0.1, 0.001}
     surrogate_steps  cat                                    {50, 200, 400}
             lambda1  cat                               {0.1, 0.001, 1e-05}
             lambda2  cat                               {1e-05, 0.1, 0.001}
    sel_n_estimators  cat                                             {400}
   sel_learning_rate  cat                            {0.03, 0.3, 0.1, 0.01}
       sel_max_depth  cat                                     {5, 12, 8, 3}
       sel_subsample  cat                                   {1.0, 0.6, 0.8}
       sel_colsample  cat                              {0.6, 1.0, 0.3, 0.1}
sel_min_child_weight  cat                                             {1.0}
      sel_reg_lambda  cat                                             {3.0}
        weight_decay  cat                                            {0.01}
           out_ratio  cat                                   {1.0, 0.1, 0.3}
    bottleneck_ratio  cat                                            {0.05}
  [FASTT-Boosted  (15 free of 15)]
        parameter kind                                   choices / range
   transform_type  cat {identity, low_rank, linear, nonlinear, diagonal}
          lambda1  cat                                           {0.001}
          lambda2  cat                               {0.001, 1e-05, 0.1}
        out_ratio  cat                                             {0.1}
 bottleneck_ratio  cat                                 {0.15, 0.5, 0.05}
       num_rounds  cat                                      {6, 1, 4, 2}
        num_trees  cat                                    {24, 8, 16, 4}
            depth  cat                                               {7}
            lmbda  cat                        {0.01, 0.0001, 1.0, 1e-06}
           lmbda2  cat                                           {1e-07}
          adam_lr  cat                                          {0.0003}
adam_weight_decay  cat                        {0.001, 1e-07, 1e-05, 0.1}
       batch_size  cat                                  {512, 128, 2048}
         patience  cat                                      {40, 20, 10}
   t_weight_decay  cat                                           {1e-06}
  [Raw XGBoost (AdaSTT)  (7 free of 7)]
           parameter kind        choices / range
    sel_n_estimators  cat   {1000, 400, 150, 50}
   sel_learning_rate  cat {0.3, 0.1, 0.03, 0.01}
       sel_max_depth  cat                    {3}
       sel_subsample  cat        {1.0, 0.8, 0.6}
       sel_colsample  cat                  {1.0}
sel_min_child_weight  cat                 {16.0}
      sel_reg_lambda  cat       {3.0, 0.3, 30.0}
  [Raw SDT  (14 free of 15)]
        parameter kind              choices / range
   t_weight_decay  cat                      {1e-06}
          lambda1  cat                      {0.001}
          lambda2  cat                      {0.001}
        out_ratio  cat                        {0.1}
 bottleneck_ratio  cat                       {0.05}
       num_rounds  cat                 {1, 2, 4, 6}
        num_trees  cat                          {8}
            depth  cat                 {2, 7, 5, 3}
            lmbda  cat   {1e-06, 1.0, 0.0001, 0.01}
           lmbda2  cat                        {0.1}
          adam_lr  cat {0.001, 0.003, 0.0003, 0.01}
adam_weight_decay  cat                      {1e-07}
       batch_size  cat             {2048, 512, 128}
         patience  cat              

## 2. Data

The three experts' pooled encoder features, their transcripts, and the transcripts' word-error counters against
the reference, for every clip. A sample of the stored counters is recomputed from the stored transcripts as a
check.

In [4]:
exp.load()

9,678 utterances x 3 routable model(s), from cache .fastt-cache/store_450a6b89356426ae
texts for 3 model(s), from cache
100.0% of 1,200 (utterance, model) counters reproduce exactly


EnsembleStore(voxpopuli, 'mean', 9,678 utterances, 3 routable)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every method, ten configurations are drawn at random
from its space (Table 2), each fitted on 70 % of the hold-out and scored on the remaining 30 %; the configuration
with the lowest corpus WER is kept. When several tie on it, the one the paper's run kept is used
(`fastt.configs.SELECTED`), after checking that it is among the tied best. The other 75 % of the clips are never
seen by the search.

In [5]:
exp.search()


=== best fastt_alternating   corpus WER 0.1873   gap closed +15.9%   (10 trials, 0.0 min)
{
  "bottleneck_ratio": 0.05,
  "num_iterations": 3,
  "out_ratio": 1.0,
  "sel_colsample": 0.3,
  "sel_learning_rate": 0.01,
  "sel_max_depth": 3,
  "sel_min_child_weight": 1.0,
  "sel_n_estimators": 400,
  "sel_reg_lambda": 3.0,
  "sel_subsample": 0.8,
  "surrogate_lr": 0.1,
  "surrogate_steps": 200,
  "transform_lr": 0.0001,
  "transform_steps": 400,
  "transform_type": "nonlinear",
  "weight_decay": 0.01
}

=== best fastt_boosted   corpus WER 0.1879   gap closed +14.1%   (10 trials, 0.0 min)
{
  "adam_lr": 0.0003,
  "adam_weight_decay": 1e-07,
  "batch_size": 512,
  "bottleneck_ratio": 0.15,
  "depth": 7,
  "epochs": 200,
  "lmbda": 1e-06,
  "lmbda2": 1e-07,
  "num_rounds": 2,
  "num_trees": 16,
  "out_ratio": 0.1,
  "patience": 20,
  "t_weight_decay": 1e-06,
  "transform_type": "low_rank"
}

=== best adastt   corpus WER 0.1920   gap closed +0.0%   (10 trials, 0.0 min)
{
  "sel_colsample": 1.


=== best rover_w   corpus WER 0.1894   gap closed +9.0%   (10 trials, 0.0 min)
{
  "alpha": 1.0,
  "conf_source": "cn",
  "null_penalty": 0.5,
  "weight_by_conf": false,
  "weighting": "log_odds"
}

=== best rover_conf   corpus WER 0.2022   gap closed -34.2%   (10 trials, 0.0 min)
{
  "alpha": 0.25,
  "conf_source": "cn",
  "conf_temp": 0.25,
  "null_penalty": 0.25,
  "weight_by_conf": false,
  "weighting": "inverse_wer"
}

=== best cn_mbr   corpus WER 0.2095   gap closed -58.9%   (10 trials, 0.0 min)
{
  "candidates": "members",
  "conf_source": "blend",
  "decode": "hyp",
  "null_prior": 0.25,
  "prior": "inverse_wer",
  "temperature": 10.0,
  "use_conf": true
}

=== best score_fusion   corpus WER 0.2167   gap closed -83.2%   (10 trials, 0.0 min)
{
  "mode": "hard",
  "w_conf": 1.0,
  "w_consensus": -1.0,
  "w_length": -1.0
}

=== best argmin_clf   corpus WER 0.1892   gap closed +9.4%   (10 trials, 0.0 min)
{
  "alpha": 0.1
}

=== best probe   corpus WER 0.1920   gap closed +0.0%   

partition,n_utts,share
tune,2420,0.2501
eval,7258,0.7499
fit,1694,0.1750
val,726,0.0750


  on the tuning-score rows: bsm(espnet_transducer) 0.1920   oracle 0.1624   gap 0.0296
  the 7,258 evaluation rows are never seen by a trial
  objectives   : corpus_wer (minimize) — crossval.METRICS['corpus_wer'], on the tuning-score rows
  14 arms x 10 trials, read from the Hub; 7,258 evaluation clips the search never saw


HPTRun('hpt_voxpopuli_30f018c8', 6 arm(s), best fastt_alternating 0.1873, eval on 7,258 rows)

In [6]:
exp.dataset()

statistic,voxpopuli
Total clips,9678
Features,1793
Hyperparameter hold-out,2420
Evaluation rows (5x2),7258
Train size (per fold),3629
Test size (per fold),3629
Hours (est.),13.6349
Reference words,246177


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
FASTT-Alternating,10,0.1873,"{""bottleneck_ratio"": 0.05, ""num_iterations"": 3, ""out_ratio"": 1.0, ""sel_colsample"": 0.3, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 1.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 3.0, ""sel_subsample"": 0.8, ""surrogate_lr"": 0.1, ""surrogate_steps"": 200, ""transform_lr"": 0.0001, ""transform_steps"": 400, ""transform_type"": ""nonlinear"", ""weight_decay"": 0.01}"
FASTT-Boosted,10,0.1879,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""bottleneck_ratio"": 0.15, ""depth"": 7, ""epochs"": 200, ""lmbda"": 1e-06, ""lmbda2"": 1e-07, ""num_rounds"": 2, ""num_trees"": 16, ""out_ratio"": 0.1, ""patience"": 20, ""t_weight_decay"": 1e-06, ""transform_type"": ""low_rank""}"
Raw XGBoost (AdaSTT),10,0.1920,"{""sel_colsample"": 1.0, ""sel_learning_rate"": 0.03, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 400, ""sel_reg_lambda"": 0.3, ""sel_subsample"": 0.6}"
Raw SDT,10,0.1920,"{""adam_lr"": 0.001, ""adam_weight_decay"": 1e-07, ""batch_size"": 512, ""depth"": 5, ""epochs"": 200, ""lmbda"": 0.0001, ""lmbda2"": 0.1, ""num_rounds"": 2, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
SelectKBest XGBoost,10,0.1920,"{""k"": 512, ""sel_colsample"": 1.0, ""sel_learning_rate"": 0.01, ""sel_max_depth"": 3, ""sel_min_child_weight"": 16.0, ""sel_n_estimators"": 1000, ""sel_reg_lambda"": 30.0, ""sel_subsample"": 0.8}"
SelectKBest SDT,10,0.1918,"{""adam_lr"": 0.0003, ""adam_weight_decay"": 1e-07, ""batch_size"": 2048, ""depth"": 5, ""epochs"": 200, ""k"": 32, ""lmbda"": 1e-06, ""lmbda2"": 0.1, ""num_rounds"": 1, ""num_trees"": 8, ""patience"": 20, ""transform_type"": ""identity""}"
ROVER (weighted),10,0.1894,"{""alpha"": 1.0, ""conf_source"": ""cn"", ""null_penalty"": 0.5, ""weight_by_conf"": false, ""weighting"": ""log_odds""}"
ROVER (confidence-weighted),10,0.2022,"{""alpha"": 0.25, ""conf_source"": ""cn"", ""conf_temp"": 0.25, ""null_penalty"": 0.25, ""weight_by_conf"": false, ""weighting"": ""inverse_wer""}"
CN-MBR,10,0.2095,"{""candidates"": ""members"", ""conf_source"": ""blend"", ""decode"": ""hyp"", ""null_prior"": 0.25, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": true}"
Score-Fusion,10,0.2167,"{""mode"": ""hard"", ""w_conf"": 1.0, ""w_consensus"": -1.0, ""w_length"": -1.0}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every method is trained
on one half and tested on the other, then the other way round: ten folds, the same for every method.

In [8]:
exp.cross_validate();

  runlog HIT  cv5x2 seed 11 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 22 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 33 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 44 — 15/15 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 55 — 15/15 group(s) cached, nothing refitted
5 repetitions x 2 folds x 24 arms, read from the Hub


## 5. Main results (Table 3)

Corpus WER and mean utterance WER (mean ± sd over the ten folds), the share of the oracle gap closed and the
selection accuracy at 0, 1 and 2 words of tolerance, and how often each method picks each expert.

In [9]:
exp.results()

Table 3 — main results, VoxPopuli
mean ± sd over 10 folds; GC / uGC / Acc in %; dist = share of clips routed to ESPnet Conformer-RNN-T / ESPnet Transformer / data2vec-audio-base; best non-oracle value per column in bold
  [individual]
                                       arm             WER            uWER         GC@0          GC@1          GC@2        uGC@0         uGC@1         uGC@2      Acc@0      Acc@1      Acc@2           dist
                    ESPnet Conformer-RNN-T 0.1761 ± 0.0019 0.1792 ± 0.0018    0.0 ± 0.0     0.0 ± 0.0     0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0     0.0 ± 0.0 63.8 ± 0.8 82.1 ± 0.4 92.5 ± 0.2 1.00-0.00-0.00
                        ESPnet Transformer 0.2372 ± 0.0012 0.2420 ± 0.0010 -228.8 ± 6.5 -313.1 ± 10.1 -537.3 ± 19.8 -207.7 ± 5.9 -311.5 ± 10.4 -610.2 ± 25.1 25.1 ± 0.7 45.9 ± 0.4 64.1 ± 0.5 0.00-1.00-0.00
                       data2vec-audio-base 0.1775 ± 0.0011 0.1847 ± 0.0011   -5.2 ± 3.3    -7.2 ± 4.5   -12.3 ± 7.7  -18.1 ± 3.0   -27.1 ± 4.3   -53.1 ± 8.6 60.9 ± 0.2 81.8 ± 0.4 92.1 ± 0.3 0.00-0.00-1.00
Best-Single-Model (ESPnet Conformer-RNN-T) 0.1761 ± 0.0019 0.1792 ± 0.0018    0.0 ± 0.0     0.0 ± 0.0     0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0     0.0 ± 0.0 63.8 ± 0.8 82.1 ± 0.4 92.5 ± 0.2 1.00-0.00-0.00
  [external]
                              arm             WER            uWER         GC@0         GC@1          GC@2        uGC@0         uGC@1         uGC@2      Acc@0      Acc@1      Acc@2           dist
                  Random-Ensemble 0.1967 ± 0.0016 0.2015 ± 0.0017  -76.9 ± 4.4 -105.2 ± 5.8 -180.5 ± 10.6  -73.8 ± 5.3  -110.7 ± 7.2 -216.8 ± 14.0 50.3 ± 0.6 70.1 ± 0.6 83.1 ± 0.5 0.33-0.33-0.33
                 ROVER (weighted) 0.1737 ± 0.0017 0.1769 ± 0.0019    9.1 ± 1.6   12.5 ± 2.3    21.4 ± 3.9    7.6 ± 1.8    11.5 ± 2.9    22.6 ± 6.0          —          —          —              —
      ROVER (confidence-weighted) 0.1840 ± 0.0017 0.1874 ± 0.0016  -29.5 ± 1.7  -40.3 ± 2.3   -69.2 ± 4.1  -27.0 ± 1.6   -40.5 ± 2.1   -79.3 ± 4.8          —          —          —              —
                           CN-MBR 0.1925 ± 0.0016 0.1965 ± 0.0016  -61.3 ± 2.7  -83.9 ± 4.0  -144.0 ± 7.5  -57.1 ± 3.3   -85.7 ± 4.7 -168.0 ± 12.1 54.9 ± 0.5 72.4 ± 0.5 83.9 ± 0.4 0.56-0.20-0.24
                     Score-Fusion 0.2066 ± 0.0016 0.2145 ± 0.0014 -114.2 ± 4.5 -156.3 ± 7.4 -268.2 ± 14.2 -116.6 ± 6.2 -175.0 ± 11.0 -343.0 ± 25.5 45.0 ± 0.8 63.5 ± 0.8 78.2 ± 0.5 0.32-0.26-0.42
         Hard-Selector (plain CE) 0.1737 ± 0.0018 0.1776 ± 0.0018    8.9 ± 1.6   12.2 ± 2.1    20.9 ± 3.5    5.5 ± 2.0     8.2 ± 3.0    16.0 ± 5.9 66.0 ± 0.8 83.8 ± 0.3 93.4 ± 0.2 0.82-0.01-0.17
Hard-Selector (cost-sensitive CE) 0.1761 ± 0.0019 0.1792 ± 0.0018    0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0    0.0 ± 0.0     0.0 ± 0.0     0.0 ± 0.0 63.8 ± 0.8 82.1 ± 0.4 92.5 ± 0.2 1.00-0.00-0.00
                        QE-Router 0.1795 ± 0.0014 0.1863 ± 0.0014  -12.6 ± 3.0  -17.2 ± 4.3   -29.6 ± 7.5  -23.4 ± 3.4   -35.1 ± 5.2  -68.8 ± 10.6 60.2 ± 0.7 79.7 ± 0.4 90.9 ± 0.5 0.42-0.15-0.43
  [classical]
                 arm             WER            uWER      GC@0      GC@1       GC@2     uGC@0     uGC@1      uGC@2      Acc@0      Acc@1      Acc@2           dist
Raw XGBoost (AdaSTT) 0.1745 ± 0.0019 0.1776 ± 0.0017 6.2 ± 2.5 8.5 ± 3.4 14.6 ± 5.8 5.3 ± 2.3 7.9 ± 3.4 15.4 ± 6.6 65.3 ± 0.7 83.7 ± 0.6 93.2 ± 0.4 0.61-0.00-0.39
             Raw SDT 0.1761 ± 0.0019 0.1792 ± 0.0018 0.0 ± 0.0 0.0 ± 0.0  0.0 ± 0.0 0.0 ± 0.0 0.0 ± 0.0  0.0 ± 0.0 63.8 ± 0.8 82.1 ± 0.4 92.5 ± 0.2 1.00-0.00-0.00
 SelectKBest XGBoost 0.1751 ± 0.0022 0.1784 ± 0.0021 3.8 ± 3.1 5.2 ± 4.2  9.0 ± 7.2 2.7 ± 2.2 4.1 ± 3.3  8.0 ± 6.4 64.7 ± 0.9 83.1 ± 0.8 93.0 ± 0.4 0.74-0.00-0.26
     SelectKBest SDT 0.1761 ± 0.0019 0.1792 ± 0.0018 0.0 ± 0.0 0.0 ± 0.0  0.0 ± 0.0 0.0 ± 0.0 0.0 ± 0.0  0.0 ± 0.0 63.8 ± 0.8 82.1 ± 0.4 92.5 ± 0.2 1.00-0.00-0.00
  [fastt]
              arm             WER            uWER       GC@0       GC@1       GC@2     uGC@0      uGC@1       uGC@2      Acc@0      A

In [10]:
exp.routing()

model,n_picks,n_fused,share_espnet_transducer,share_espnet_transformer,share_data2vec,top_member,share_top,entropy,entropy_max,share_sd_mean,switch_rate
achievable_oracle,36290,0,0.9248,0.0079,0.0674,espnet_transducer,0.9248,0.4215,1.5850,0.0017,0.0752
adastt,36290,0,0.6063,0.0000,0.3937,espnet_transducer,0.6063,0.9672,1.5850,0.0724,0.3937
argmin_clf,36290,0,0.8235,0.0086,0.1679,espnet_transducer,0.8235,0.7220,1.5850,0.0090,0.1765
bsm,36290,0,1.0000,0.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.0000
cn_mbr,36290,0,0.5572,0.2005,0.2424,espnet_transducer,0.5572,1.4305,1.5850,0.0116,0.4428
data2vec,36290,0,0.0000,0.0000,1.0000,data2vec,1.0000,0.0000,1.5850,0.0000,1.0000
espnet_transducer,36290,0,1.0000,0.0000,0.0000,espnet_transducer,1.0000,0.0000,1.5850,0.0000,0.0000
espnet_transformer,36290,0,0.0000,1.0000,0.0000,espnet_transformer,1.0000,0.0000,1.5850,0.0000,1.0000
fastt_alternating,36290,0,0.4836,0.0000,0.5164,data2vec,0.5164,0.9992,1.5850,0.0161,0.5164
fastt_boosted,36290,0,0.3508,0.0000,0.6492,data2vec,0.6492,0.9348,1.5850,0.0220,0.6492


## 6. Significance (Table 6)

Every method against the FASTT anchor (the better of the two FASTT variants on corpus WER) on the same ten folds:
Dietterich's 5×2 cv paired $t$ test and Alpaydın's combined 5×2 cv $F$ test, on corpus WER and on mean utterance
WER.

In [11]:
exp.tests()

Table 6 — every arm against the FASTT anchor, VoxPopuli
on the same 10 folds; positive effect = the anchor is better, in the metric's own units; p < 0.05 in bold. The F-test is the one the paper reports.
  [mean utterance WER — anchor FASTT-Alternating]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
           ESPnet Conformer-RNN-T +0.0026        4.356       0.0073     10.456     0.0091
               ESPnet Transformer +0.0654       61.142       0.0000   3570.531     0.0000
              data2vec-audio-base +0.0080        9.956       0.0002     71.393     0.0001
                Best-Single-Model +0.0026        4.356       0.0073     10.456     0.0091
                  Random-Ensemble +0.0249       26.549       0.0000    742.540     0.0000
                 ROVER (weighted) +0.0002        1.031       0.3498      1.386     0.3778
      ROVER (confidence-weighted) +0.0107       18.476       0.0000    278.358     0.0000
                           CN-MBR +0.0198       23.691       0.0000    492.726     0.0000
                     Score-Fusion +0.0378       15.574       0.0000    214.410     0.0000
         Hard-Selector (plain CE) +0.0009        2.522       0.0530      1.731     0.2832
Hard-Selector (cost-sensitive CE) +0.0026        4.356       0.0073     10.456     0.0091
                        QE-Router +0.0096       10.374       0.0001     79.087     0.0001
             Raw XGBoost (AdaSTT) +0.0010        3.357       0.0202      3.812     0.0763
                          Raw SDT +0.0026        4.356       0.0073     10.456     0.0091
              SelectKBest XGBoost +0.0017        3.865       0.0118      6.007     0.0307
                  SelectKBest SDT +0.0026        4.356       0.0073     10.456     0.0091
                    FASTT-Boosted +0.0003        1.451       0.2066      4.267     0.0614
            Oracle (no tolerance) -0.0277      -25.870       0.0000    729.999     0.0000
        Oracle (1 word tolerance) -0.0176      -11.830       0.0001    167.276     0.0000
        Oracle (2 word tolerance) -0.0077       -5.609       0.0025     54.362     0.0002
  [corpus WER — anchor FASTT-Alternating]
                       comparator  effect t_dietterich p_dietterich F_alpaydin p_alpaydin
           ESPnet Conformer-RNN-T +0.0028        4.799       0.0049     11.693     0.0071
               ESPnet Transformer +0.0639       68.134       0.0000   4414.485     0.0000
              data2vec-audio-base +0.0042        4.998       0.0041     11.714     0.0071
                Best-Single-Model +0.0028        4.799       0.0049     11.693     0.0071
                  Random-Ensemble +0.0233       31.863       0.0000    917.769     0.0000
                 ROVER (weighted) +0.0004        2.173       0.0818      1.400     0.3730
      ROVER (confidence-weighted) +0.0107       15.548       0.0000    187.912     0.0000
                           CN-MBR +0.0192       21.172       0.0000    370.642     0.0000
                     Score-Fusion +0.0333       19.045       0.0000    331.283     0.0000
         Hard-Selector (plain CE) +0.0004        1.950       0.1087      0.870     0.6035
Hard-Selector (cost-sensitive CE) +0.0028        4.799       0.0049     11.693     0.0071
                        QE-Router +0.0061        5.330       0.0031     15.824     0.0035
             Raw XGBoost (AdaSTT) +0.0011        3.127       0.0261      2.554     0.1562
                          Raw SDT +0.0028        4.799       0.0049     11.693     0.0071
              SelectKBest XGBoost +0.0018        3.760       0.0132      4.508     0.0551
                  SelectKBest SDT +0.0028        4.799       0.0049     11.693     0.0071
                    FASTT-Boosted +0.0001        1.822       0.1280      1.278     0.4151
            Oracle (no tolerance) -0.0239      -21.800       0.0000    543.467     0.0000
        Oracle (1 word tolerance) -0.0167      -10.959       0.0001    152.505     0.0000
        Oracle

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_voxpopuli.json`;
`manuscript_tables` reads it from the folder `FASTT_RECORDS_DIR` names.

In [12]:
exp.save();

wrote main_voxpopuli.json
